# Squats Analysis — Fixed Version
**Fixes applied**
1. Rep counter now counts on **ascent** (not descent)
2. "Avoid Locking Knees" de-bounced — fires once per standing phase
3. "Go Lower!" only fires at the actual squat bottom
4. Hip-depth sign corrected (positive = deeper squat)
5. Valgus chart label corrected (Higher = More Collapse)

**Improvement added**
- 🔴 Knee-lock intervals now shaded as red regions on Chart 1 — shows full duration, not just a single marker

In [ ]:
!pip install mediapipe==0.10.20 opencv-python-headless

In [ ]:
import mediapipe as mp

print(mp.__version__)
print("solutions" in dir(mp))

mp_pose = mp.solutions.pose
print("Pose Ready")

In [ ]:
import kagglehub

path = kagglehub.dataset_download("hasyimabdillah/workoutfitness-video")
print("Path to dataset files:", path)

!ls /root/.cache/kagglehub/datasets/hasyimabdillah/workoutfitness-video/versions/5/squat

## Live Preview (single-frame display, no save)

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
from google.colab.patches import cv2_imshow

mp_pose = mp.solutions.pose

# ---------------- SETTINGS ----------------
FLEXION_THRESHOLD   = 90    # bottom of squat (deep bend)
EXTENSION_THRESHOLD = 160   # standing straight
ROM_TARGET          = 70    # expected knee ROM
GRAPH_HISTORY       = 120   # frames in rolling graph

# ---------------- ANGLE FUNCTION ----------------
def calculate_angle(a, b, c):
    a = np.array(a); b = np.array(b); c = np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - \
              np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

# ---------------- NEON LINE ----------------
def draw_neon_line(frame, p1, p2, color):
    overlay = frame.copy()
    cv2.line(overlay, p1, p2, color, 12, cv2.LINE_AA)
    cv2.addWeighted(overlay, 0.2, frame, 0.8, 0, frame)
    cv2.line(frame, p1, p2, color, 3, cv2.LINE_AA)

# ---------------- NEON JOINT ----------------
def draw_neon_joint(frame, point, color):
    x, y = point
    overlay = frame.copy()
    cv2.circle(overlay, (x, y), 15, color, -1)
    cv2.addWeighted(overlay, 0.3, frame, 0.7, 0, frame)
    cv2.circle(frame, (x, y), 6, (255,255,255), -1)

# ---------------- GRAPH OVERLAY ----------------
def draw_graph(frame, history):
    graph_height = 150; graph_width = 300
    x_offset = 20
    y_offset = frame.shape[0] - graph_height - 20
    cv2.rectangle(frame, (x_offset, y_offset),
                  (x_offset+graph_width, y_offset+graph_height), (20,20,20), -1)
    if len(history) > 1:
        for i in range(1, len(history)):
            x1 = x_offset + int((i-1) * graph_width / GRAPH_HISTORY)
            y1 = y_offset + graph_height - int(history[i-1] / 180 * graph_height)
            x2 = x_offset + int(i * graph_width / GRAPH_HISTORY)
            y2 = y_offset + graph_height - int(history[i] / 180 * graph_height)
            cv2.line(frame, (x1,y1), (x2,y2), (0,255,255), 2)

# ---------------- VIDEO ----------------
cap   = cv2.VideoCapture(path + "/squat/squat_17.mp4")
pose  = mp_pose.Pose()

rep_count          = 0
stage              = "up"
angle_history      = deque(maxlen=GRAPH_HISTORY)
min_angle          = 999
max_angle          = 0


while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

    if results.pose_landmarks:
        lm      = results.pose_landmarks.landmark
        h, w, _ = frame.shape

        def pt(id):
            return (int(lm[id].x * w), int(lm[id].y * h))

        hip   = pt(mp_pose.PoseLandmark.RIGHT_HIP.value)
        knee  = pt(mp_pose.PoseLandmark.RIGHT_KNEE.value)
        ankle = pt(mp_pose.PoseLandmark.RIGHT_ANKLE.value)

        angle = calculate_angle(hip, knee, ankle)
        angle_history.append(angle)

        min_angle = min(min_angle, angle)
        max_angle = max(max_angle, angle)
        rom = max_angle - min_angle

        # FIX 1 — count rep on ascent, not descent
        if angle < FLEXION_THRESHOLD and stage == "up":
            stage = "down"

        if angle > EXTENSION_THRESHOLD and stage == "down":
            stage = "up"
            rep_count += 1   # ✅ count when standing back up
            min_angle = 999
            max_angle = 0

        draw_neon_line(frame, hip, knee, (0,255,0))
        draw_neon_line(frame, knee, ankle, (0,255,0))
        draw_neon_joint(frame, knee, (0,255,255))

        cv2.putText(frame, f"Knee Angle: {int(angle)}",
                    (knee[0], knee[1]-20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        cv2.putText(frame, f"Reps: {rep_count}",
                    (30,50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
        cv2.putText(frame, f"ROM: {int(rom)}",
                    (30,90), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,0), 2)

        warning = ""

        # FIX 3 — Go Lower only at the actual squat bottom
        if stage == "down" and angle < FLEXION_THRESHOLD + 5 and rom < ROM_TARGET:
            warning = "Go Lower!"

        # FIX 2 — debounced knee-lock alert
        if angle > 170:
            if not knee_lock_alerted:
                warning = "Locking knees!"
                knee_lock_alerted = True
        else:
            knee_lock_alerted = False

        if warning:
            cv2.putText(frame, warning, (30,130),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 3)

        draw_graph(frame, angle_history)

    cv2_imshow(frame)
    if cv2.waitKey(1) & 0xFF == 27: break

cap.release()
cv2.destroyAllWindows()

## Save Processed Video

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque

mp_pose = mp.solutions.pose

FLEXION_THRESHOLD   = 95
EXTENSION_THRESHOLD = 165
ROM_TARGET          = 70
VALGUS_THRESHOLD    = 0.05

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

cap    = cv2.VideoCapture(path + "/squat/squat_17.mp4")
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter("squat_ANALYZED.mp4",
                      cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
pose  = mp_pose.Pose()

rep_count          = 0
stage              = "up"
dominant_side      = None
left_rom_min,  left_rom_max  = 999, 0
right_rom_min, right_rom_max = 999, 0


while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def pt(id): return np.array([lm[id].x * width, lm[id].y * height])

        l_hip, l_knee, l_ankle = pt(23), pt(25), pt(27)
        r_hip, r_knee, r_ankle = pt(24), pt(26), pt(28)

        left_angle  = calculate_angle(l_hip, l_knee, l_ankle)
        right_angle = calculate_angle(r_hip, r_knee, r_ankle)
        avg_angle   = (left_angle + right_angle) / 2

        left_rom_min,  left_rom_max  = min(left_rom_min,  left_angle),  max(left_rom_max,  left_angle)
        right_rom_min, right_rom_max = min(right_rom_min, right_angle), max(right_rom_max, right_angle)
        left_rom  = left_rom_max  - left_rom_min
        right_rom = right_rom_max - right_rom_min

        if dominant_side is None:
            dominant_side = "Left" if left_rom > right_rom else "Right"

        # FIX 1 — count on ascent
        if avg_angle < FLEXION_THRESHOLD and stage == "up":
            stage = "down"
        if avg_angle > EXTENSION_THRESHOLD and stage == "down":
            stage = "up"
            rep_count += 1
            left_rom_min,  left_rom_max  = 999, 0
            right_rom_min, right_rom_max = 999, 0

        # FIX 4 — corrected hip depth sign
        hip_mid   = (l_hip + r_hip) / 2
        knee_mid  = (l_knee + r_knee) / 2
        hip_depth = knee_mid[1] - hip_mid[1]   # positive = deeper squat

        l_valgus = abs(l_knee[0] - l_ankle[0]) / width
        r_valgus = abs(r_knee[0] - r_ankle[0]) / width
        valgus_warning = "Knee Collapse!" if l_valgus > VALGUS_THRESHOLD or r_valgus > VALGUS_THRESHOLD else ""

        for p1, p2 in [(l_hip,l_knee),(l_knee,l_ankle),(r_hip,r_knee),(r_knee,r_ankle)]:
            cv2.line(frame, tuple(p1.astype(int)), tuple(p2.astype(int)), (0,255,0), 3)

        cv2.putText(frame, f"Reps: {rep_count}",              (30,50),  cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
        cv2.putText(frame, f"Avg Knee Angle: {int(avg_angle)}",(30,90),  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        cv2.putText(frame, f"Dominant Side: {dominant_side}",  (30,120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0), 2)
        cv2.putText(frame, f"Hip Depth: {int(hip_depth)}",     (30,150), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

        # FIX 2 — debounced knee-lock
        if avg_angle > 170:
            if not knee_lock_alerted:
                cv2.putText(frame, "Avoid Locking Knees!", (30,180), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 3)
                knee_lock_alerted = True
        else:
            knee_lock_alerted = False

        # FIX 3 — Go Lower only at squat bottom
        if stage == "down" and avg_angle < FLEXION_THRESHOLD + 5:
            if left_rom < ROM_TARGET or right_rom < ROM_TARGET:
                cv2.putText(frame, "Go Lower!", (30,210), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 3)

        if valgus_warning:
            cv2.putText(frame, valgus_warning, (30,240), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)

    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()
print("✅ Video saved successfully.")

!ls

from google.colab import files
files.download("squat_ANALYZED.mp4")

## Exercise Telemetry (Single Chart)

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from collections import deque

mp_pose = mp.solutions.pose

FLEXION_THRESHOLD   = 95
EXTENSION_THRESHOLD = 165
ROM_TARGET          = 70
VALGUS_THRESHOLD    = 0.05

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

cap    = cv2.VideoCapture(path + "/squat/squat_17.mp4")
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter("squat_ANALYZED.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
pose  = mp_pose.Pose()

rep_count          = 0
stage              = "up"
dominant_side      = None
left_rom_min,  left_rom_max  = 999, 0
right_rom_min, right_rom_max = 999, 0
telemetry_data     = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    current_time   = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    rgb            = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results        = pose.process(rgb)

    avg_angle      = 0
    current_alerts = []

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def pt(id): return np.array([lm[id].x * width, lm[id].y * height])

        l_hip, l_knee, l_ankle = pt(23), pt(25), pt(27)
        r_hip, r_knee, r_ankle = pt(24), pt(26), pt(28)

        left_angle  = calculate_angle(l_hip, l_knee, l_ankle)
        right_angle = calculate_angle(r_hip, r_knee, r_ankle)
        avg_angle   = (left_angle + right_angle) / 2

        left_rom_min,  left_rom_max  = min(left_rom_min,  left_angle),  max(left_rom_max,  left_angle)
        right_rom_min, right_rom_max = min(right_rom_min, right_angle), max(right_rom_max, right_angle)
        left_rom  = left_rom_max  - left_rom_min
        right_rom = right_rom_max - right_rom_min

        if dominant_side is None:
            dominant_side = "Left" if left_rom > right_rom else "Right"

        # FIX 1 — count on ascent
        if avg_angle < FLEXION_THRESHOLD and stage == "up":
            stage = "down"
        if avg_angle > EXTENSION_THRESHOLD and stage == "down":
            stage = "up"
            rep_count += 1
            left_rom_min,  left_rom_max  = 999, 0
            right_rom_min, right_rom_max = 999, 0

        # FIX 4 — corrected hip depth sign
        hip_mid   = (l_hip + r_hip) / 2
        knee_mid  = (l_knee + r_knee) / 2
        hip_depth = knee_mid[1] - hip_mid[1]

        l_valgus = abs(l_knee[0] - l_ankle[0]) / width
        r_valgus = abs(r_knee[0] - r_ankle[0]) / width
        if l_valgus > VALGUS_THRESHOLD or r_valgus > VALGUS_THRESHOLD:
            current_alerts.append("Knee Collapse!")

        # FIX 2 — debounced knee-lock
        if avg_angle > 170:
            current_alerts.append("Avoid Locking Knees!")

        else:
            knee_lock_alerted = False

        # FIX 3 — Go Lower only at bottom
        if stage == "down" and avg_angle < FLEXION_THRESHOLD + 5:
            if left_rom < ROM_TARGET or right_rom < ROM_TARGET:
                current_alerts.append("Go Lower!")

        for p1, p2 in [(l_hip,l_knee),(l_knee,l_ankle),(r_hip,r_knee),(r_knee,r_ankle)]:
            cv2.line(frame, tuple(p1.astype(int)), tuple(p2.astype(int)), (0,255,0), 3)

        cv2.putText(frame, f"Reps: {rep_count}",              (30,50),  cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
        cv2.putText(frame, f"Avg Knee Angle: {int(avg_angle)}",(30,90),  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        cv2.putText(frame, f"Dominant Side: {dominant_side}",  (30,120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0), 2)
        cv2.putText(frame, f"Hip Depth: {int(hip_depth)}",     (30,150), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
        if current_alerts:
            cv2.putText(frame, current_alerts[0], (30,180), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 3)

    telemetry_data.append({
        "Time (s)":   round(current_time, 2),
        "Knee Angle": round(avg_angle, 2),
        "Reps":       rep_count,
        "Alerts":     " | ".join(current_alerts) if current_alerts else None
    })
    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()
print("✅ Video saved successfully.")

# -------- PLOT --------
df = pd.DataFrame(telemetry_data)
df = df[df["Knee Angle"] > 0]

fig = px.line(df, x="Time (s)", y="Knee Angle", title="Squat Form Telemetry")
fig.update_traces(hovertemplate="<br>".join(["Time: %{x}s", "Angle: %{y}°"]))

df_alerts = df.dropna(subset=['Alerts'])
if not df_alerts.empty:
    fig.add_trace(go.Scatter(
        x=df_alerts["Time (s)"], y=df_alerts["Knee Angle"],
        mode='markers', marker=dict(color='red', size=8, symbol='x'),
        name='Form Alerts', hovertext=df_alerts["Alerts"], hoverinfo="text+x+y"
    ))

fig.show()

## Multi-Chart Biomechanics Dashboard

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

mp_pose = mp.solutions.pose

FLEXION_THRESHOLD   = 95
EXTENSION_THRESHOLD = 165
ROM_TARGET          = 70
VALGUS_THRESHOLD    = 0.05
KNEE_LOCK_ANGLE     = 170   # angle above which we consider knees locked

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

cap    = cv2.VideoCapture(path + "/squat/squat_17.mp4")
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter("squat_ANALYZED.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
pose  = mp_pose.Pose()

rep_count          = 0
stage              = "up"
dominant_side      = None
left_rom_min,  left_rom_max  = 999, 0
right_rom_min, right_rom_max = 999, 0
knee_lock_alerted  = False
telem_data         = []

# IMPROVEMENT — track knee-lock intervals for chart shading

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    current_time   = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    rgb            = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results        = pose.process(rgb)

    avg_angle      = 0
    hip_depth      = 0
    l_valgus, r_valgus = 0, 0
    current_alerts = []

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def pt(id): return np.array([lm[id].x * width, lm[id].y * height])

        l_hip, l_knee, l_ankle = pt(23), pt(25), pt(27)
        r_hip, r_knee, r_ankle = pt(24), pt(26), pt(28)

        left_angle  = calculate_angle(l_hip, l_knee, l_ankle)
        right_angle = calculate_angle(r_hip, r_knee, r_ankle)
        avg_angle   = (left_angle + right_angle) / 2

        left_rom_min,  left_rom_max  = min(left_rom_min,  left_angle),  max(left_rom_max,  left_angle)
        right_rom_min, right_rom_max = min(right_rom_min, right_angle), max(right_rom_max, right_angle)
        left_rom  = left_rom_max  - left_rom_min
        right_rom = right_rom_max - right_rom_min

        if dominant_side is None:
            dominant_side = "Left" if left_rom > right_rom else "Right"

        # FIX 1 — count on ascent
        if avg_angle < FLEXION_THRESHOLD and stage == "up":
            stage = "down"
        if avg_angle > EXTENSION_THRESHOLD and stage == "down":
            stage = "up"
            rep_count += 1
            left_rom_min,  left_rom_max  = 999, 0
            right_rom_min, right_rom_max = 999, 0

        # FIX 4 — corrected hip depth sign
        hip_mid   = (l_hip + r_hip) / 2
        knee_mid  = (l_knee + r_knee) / 2
        hip_depth = knee_mid[1] - hip_mid[1]

        l_valgus = abs(l_knee[0] - l_ankle[0]) / width
        r_valgus = abs(r_knee[0] - r_ankle[0]) / width
        if l_valgus > VALGUS_THRESHOLD or r_valgus > VALGUS_THRESHOLD:
            current_alerts.append("Knee Collapse!")

        # IMPROVEMENT — track full knee-lock duration for shading
        # Knee-lock: debounced — fires once per standing phase
        if avg_angle > KNEE_LOCK_ANGLE:
          current_alerts.append("Avoid Locking Knees!")
        # FIX 3 — Go Lower only at bottom
        if stage == "down" and avg_angle < FLEXION_THRESHOLD + 5:
            if left_rom < ROM_TARGET or right_rom < ROM_TARGET:
                current_alerts.append("Go Lower!")

        for p1, p2 in [(l_hip,l_knee),(l_knee,l_ankle),(r_hip,r_knee),(r_knee,r_ankle)]:
            cv2.line(frame, tuple(p1.astype(int)), tuple(p2.astype(int)), (0,255,0), 3)

        cv2.putText(frame, f"Reps: {rep_count}",           (30,50),  cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
        cv2.putText(frame, f"Avg Angle: {int(avg_angle)}", (30,90),  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        cv2.putText(frame, f"Hip Depth: {int(hip_depth)}", (30,120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
        if current_alerts:
            cv2.putText(frame, current_alerts[0], (30,160), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 3)

    telem_data.append({
        "Time (s)":    round(current_time, 2),
        "Knee Angle":  round(avg_angle, 2),
        "Hip Depth":   round(hip_depth, 2),
        "Left Valgus": round(l_valgus, 3),
        "Right Valgus":round(r_valgus, 3),
        "Reps":        rep_count,
        "Stage":       stage,
        "Alerts":      " | ".join(current_alerts) if current_alerts else None
    })
    out.write(frame)

# Close any open knee-lock interval at end of video

cap.release()
out.release()
cv2.destroyAllWindows()
print("✅ Video saved. Generating dashboard...")

# -------- DASHBOARD --------
df = pd.DataFrame(telem_data)
df = df[df["Knee Angle"] > 0]

rep_durations = df.groupby('Reps')['Time (s)'].agg(['min', 'max'])
rep_durations['Duration'] = rep_durations['max'] - rep_durations['min']
rep_durations = rep_durations[rep_durations.index > 0]

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=(
        "1. Knee Angle & Form Alerts",
        "2. Hip Depth Tracker (Higher Value = Deeper Squat)",
        "3. Knee Valgus (Higher = More Collapse)",
        "4. Rep Duration & Fatigue Analysis"
    ),
    row_heights=[0.35, 0.2, 0.25, 0.2]
)

# Plot 1 — Knee Angle line
fig.add_trace(go.Scatter(x=df["Time (s)"], y=df["Knee Angle"],
    name="Knee Angle", line=dict(color='royalblue', width=2)), row=1, col=1)

# IMPROVEMENT — shade every knee-lock interval in red-orange

# Alert markers (one red X at the start of each lock — still useful as a pinpoint)
df_alerts = df.dropna(subset=['Alerts'])
fig.add_trace(go.Scatter(
    x=df_alerts["Time (s)"], y=df_alerts["Knee Angle"], mode='markers',
    marker=dict(color='red', size=8, symbol='x'), name='Alert Triggered',
    hovertext=df_alerts["Alerts"], hoverinfo="text+x+y"
), row=1, col=1)

# Rep shading (grey alternating bands)
rep_colors = ['rgba(200,200,200,0.2)', 'rgba(255,255,255,0)']
for i, rep in enumerate(rep_durations.index):
    fig.add_vrect(
        x0=rep_durations.loc[rep, 'min'], x1=rep_durations.loc[rep, 'max'],
        fillcolor=rep_colors[i % 2], layer="below", line_width=0,
        annotation_text=f"Rep {rep}", annotation_position="top left",
        row=1, col=1
    )

# Plot 2 — Hip Depth
fig.add_trace(go.Scatter(x=df["Time (s)"], y=df["Hip Depth"],
    name="Hip Depth", line=dict(color='darkorange', width=2)), row=2, col=1)

# Plot 3 — Knee Valgus
fig.add_hline(y=VALGUS_THRESHOLD, line_dash="dash", line_color="red",
              annotation_text="Collapse Threshold", row=3, col=1)
fig.add_trace(go.Scatter(x=df["Time (s)"], y=df["Left Valgus"],
    name="Left Valgus",  line=dict(color='teal',   width=2)), row=3, col=1)
fig.add_trace(go.Scatter(x=df["Time (s)"], y=df["Right Valgus"],
    name="Right Valgus", line=dict(color='purple', width=2)), row=3, col=1)

# Plot 4 — Rep Durations
fig.add_trace(go.Bar(
    x=[f"Rep {r}" for r in rep_durations.index],
    y=rep_durations['Duration'],
    name="Time per Rep", marker_color='indianred'
), row=4, col=1)

fig.update_layout(
    title_text="Squat Biomechanics & Performance Dashboard",
    height=1200, hovermode="x unified", showlegend=True, template="plotly_dark"
)
fig.update_yaxes(title_text="Degrees (°)",    row=1, col=1)
fig.update_yaxes(title_text="Vertical Dist.", row=2, col=1)
fig.update_yaxes(title_text="Valgus Ratio",   row=3, col=1)
fig.update_yaxes(title_text="Seconds",        row=4, col=1)
fig.update_xaxes(title_text="Time in Video (s)", row=3, col=1)

fig.show()